# tinydet experiments

This notebook wires the `tinydet` package together. The logic lives in `tinydet/`; edit those files and re-run a cell, and `%autoreload 2` picks the change up without restarting the kernel.

How the search works:
1. `search/space.json` lists every hyperparameter with its range, your prior over values, and your belief in its impact. It also holds the stages, budgets, constraints and objective. Edit it at any time; it is re-read before every group.
2. `experiments/<name>/ledger.jsonl` records every trial, rung result and decision. Re-running the search cell resumes from it, including work cut off by an interrupt.
3. Each **group** trains up to `group_size` trials **in lockstep** on the same batches (they share data settings), then evaluates each one on `val_zoom` / `val_full` / `val_neg`.
4. Trials start at the first rung (15 epochs + a short decay branch). The top third moves on to 40, then 120 epochs (successive halving). Within a stage, new trials come from the stage's seeds, then from your priors, then from Optuna's TPE.
5. When a stage's budget is spent, its winner's settings are pinned for later stages.

First time on a machine? Run `python tests/smoke_test.py` from `models/` (2-5 min, synthetic data) to check the whole stack, GPU included.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import torch

sys.path.insert(0, str(Path.cwd()))  # this folder (models/) contains the tinydet package
import tinydet as td
from tinydet.config import Config

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

If autoreload ever gets confused (for example after renaming a class), run `td.reload()`, then rebuild any objects created from the old code (datasets, `ctx`, `search`).

In [ ]:
# td.reload()

## 1. Data

Use the full-resolution `~/datasets/coco` folder. The `coco-*x*` folders were made with `magick -resize WxH!`, which ignores the aspect ratio: balls become ellipses, and the zoom crops need the original resolution anyway.

The first run reads the COCO annotation files (~20 s) and caches a small index in `cache/`. Splits are built once from a fixed seed and saved, so every trial is scored on the same images.

In [ ]:
COCO_ROOT = "~/datasets/coco"
SPACE = Path("search/space.json")
EXP = Path("experiments") / "ball-v1"
CACHE = Path("cache")
WORKERS = 10          # CPU decode/augment processes; raise it if the GPU sits idle

index = td.data.load_index(COCO_ROOT, class_ids=[37], cache_dir=CACHE)
splits = td.data.build_splits(index, cache_dir=CACHE)
ctx = td.train.DataContext(index, splits, COCO_ROOT, workers=WORKERS)

In [ ]:
# Augmented training samples with their targets.
# green = positive box, orange = ignored (tiny / crowd / clipped), blue = target heatmap
cfg = Config({"data.coco_root": COCO_ROOT})
ds = td.data.train_dataset(cfg, index, splits)
td.viz.preview(ds, cfg, n=12, seed=0);

### Camera simulation and copy-paste

Two searchable additions (stage `domain` in `space.json`):

- **Camera simulation** (`tinydet/camera.py`): what the ESP32-S3's OV2640 pipeline does to a frame. Low-light shot/read noise amplified by auto-gain, exposure errors with clipped highlights, white-balance casts, ISP denoise + sharpening halos, chroma noise and row banding, lens shading and distortion, motion blur and rolling-shutter shear from the moving car, unfiltered downscaling, and RGB565. Knobs: `data.device_strength`, `data.device_motion`, `data.device_lens`.
- **Copy-paste** (`tinydet/paste.py`): balls that are large in their training image are cut out with their COCO masks once, then pasted small (6-36 px) with feathered edges, brightness matching and, with `data.paste_floor_bias`, a lower-frame placement and contact shadow. Knobs: `data.paste_prob`, `data.paste_max`, `data.paste_floor_bias`.

Frames from the car with no balls in them can go in a folder set as `data.extra_bg_dir`; they are added as backgrounds (and get pasted balls). When you know the lens, pin `data.device_lens` in `fixed`.

In [ ]:
bank = ctx.cutout_bank(20)   # first call cuts the balls out of the training images (one-off)
cfg_dev = Config({"data.coco_root": COCO_ROOT, "data.device_strength": 0.6, "data.device_motion": 0.3,
                  "data.device_lens": "mild", "data.paste_prob": 0.75, "data.paste_floor_bias": 0.5,
                  "data.bg_source": "indoor"})
td.viz.preview(td.data.train_dataset(cfg_dev, index, splits, bank=bank), cfg_dev, n=12, seed=0);

In [ ]:
# One validation window through the camera model at increasing strength
td.viz.camera_strengths(td.data.eval_dataset("val_zoom", cfg, index, splits)[0][0]);

In [ ]:
# Validation: val_zoom = balls at 6-40 px (the car's working range);
# val_dev = half of those windows through a fixed, seeded camera model
val_zoom = td.data.eval_dataset("val_zoom", cfg, index, splits)
td.viz.preview(val_zoom, cfg, n=6, cols=6, seed=1);
td.viz.preview(td.data.eval_dataset("val_dev", cfg, index, splits), cfg, n=6, cols=6, seed=1);

In [ ]:
# Is training data-bound? Compare this with the images/s the GPU needs (batch 64 x steps/s).
loader = td.data.make_loader(ds, 64, shuffle=True, workers=WORKERS)
td.data.profile_loader(loader, n_batches=40);
del loader

## 2. What each architecture costs

Static counts, no training. `pred_ms` uses the latency model in `space.json` (uncalibrated until you add measured `invoke()` timings, section 6).

In [ ]:
import itertools
import pandas as pd

space = td.search.load_space(SPACE)
rows = []
for size, width, tap, head in itertools.product([96, 120, 160], [0.35, 0.5, 0.75, 1.0], [False, True],
                                                ["basic", "dw1", "dwsep", "bottleneck"]):
    c = Config({"data.input_size": size, "model.width": width, "model.tap": tap, "model.head": head,
                "model.pretrained": False})
    rows.append({"input": size, "width": width, "tap": tap, "head": head,
                 **td.models.profile(c, space.get("latency_model"))})
zoo = pd.DataFrame(rows)
zoo.pivot_table(index=["input", "width"], columns="head", values="pred_ms", aggfunc="max")

## 3. Search

`search.status()` shows the stages. `search.run(...)` plans and trains groups until the limits are hit or every stage is closed; interrupt it whenever you like and re-run the cell to continue. The dashboard is rebuilt after every group.

In [ ]:
search = td.search.Search(EXP, SPACE, ctx, device=DEVICE, mlflow=False)
search.status()

In [ ]:
# One group first, to check timings and memory on the 3060
search.run(max_groups=1)

In [ ]:
# Then let it go. Edit search/space.json between groups if you want to steer it.
search.run(max_hours=8)

In [ ]:
search.leaderboard(20)

In [ ]:
# What the search has learned per parameter, next to your priors (also in observed.json)
obs = search.report()
pd.DataFrame({k: {"stage": v["stage"], "prior_impact": v["prior_impact"],
                  "observed_spread": round(v["observed_spread"], 4),
                  "observed_importance": v.get("observed_importance")} for k, v in obs.items()}).T

In [ ]:
from IPython.display import IFrame
dash = td.dashboard.build(EXP, SPACE)
print(dash.resolve())
IFrame(str(dash), width="100%", height=900)

In [ ]:
search.note("Started the structure stage after reviewing baselines")  # free-text notes go in the ledger

**Changing the objective.** Objectives are recomputed from stored metrics every time the ledger is read, so editing `objective.terms` in `space.json` re-scores the whole history. If a term uses a metric older trials don't have, their objective becomes empty; `search.reevaluate()` re-scores their saved checkpoints with the current evaluation code.

In [ ]:
# search.reevaluate()

## 4. A one-off run outside the staged search

`manual()` trains one config (current pinned settings + your overrides, cosine schedule) and logs it as stage `manual`, so it shows up in the leaderboard and dashboard.

In [ ]:
# res = search.manual({"model.head": "bottleneck", "data.input_size": 120}, epochs=60, note="bottleneck check")
# res

## 5. Inspect and export the best trial

Folds the trial's Platt scaling into its last conv (so the firmware's sigmoid reports calibrated probabilities for SLAM), exports to ONNX and full-INT8 TFLite (positive-heavy calibration set), scores the INT8 model on the same splits, and writes the `.tflite` + `model_metadata.json` that `firmware/build.rs` checks.

In [ ]:
best = search.leaderboard(1).iloc[0]["trial"]
model, best_cfg, ck = td.export.load_trial(EXP / "trials" / best)
best_cfg = best_cfg.override({"data.coco_root": COCO_ROOT})
print(best, best_cfg.diff())
model = model.to(DEVICE)
td.viz.predictions(model, best_cfg, td.data.eval_dataset("val_zoom", best_cfg, index, splits), n=12, device=DEVICE);

In [ ]:
out = Path("output") / best
a, b = ck["metrics"].get("val_zoom/platt_a", 1.0), ck["metrics"].get("val_zoom/platt_b", 0.0)
print(f"Platt scaling folded into the head: logit' = {a:.3f} * logit + {b:.3f}")
calibrated = td.export.apply_platt(model.cpu(), a, b)
onnx_path = td.export.export_onnx(calibrated, best_cfg, out / "model.onnx")
calib = td.export.calibration_array(best_cfg, index, splits, n=400)
tflite_path = td.export.convert_int8(onnx_path, out / "tf", calib)

int8 = td.export.evaluate_tflite(tflite_path, best_cfg, ctx)
keys = ["val_zoom/ap", "val_dev/ap", "val_zoom/f1_best", "val_full/ap", "val_full/f1", "val_neg/fp_per_img",
        "val_zoom/ece", "val_zoom/logit_min", "val_zoom/logit_max"]
# float = raw scores before Platt; int8 = the exported (calibrated) model, so its ECE should be lower
pd.DataFrame({"float (raw)": {k: ck["metrics"].get(k) for k in keys}, "int8 (calibrated)": {k: int8.get(k) for k in keys}})

In [ ]:
# Copy into the firmware (rebuild with `cargo run --release` from firmware/)
# conf_threshold is now a probability: 0.5 = "at least even odds this is a ball"
td.export.write_firmware(tflite_path, best_cfg, "../firmware/model", conf_threshold=0.5,
                         training={"trial": best, "platt": [a, b],
                                   "float_metrics": {k: v for k, v in ck["metrics"].items() if not isinstance(v, dict)},
                                   "int8_metrics": {k: v for k, v in int8.items() if not isinstance(v, dict)}})

## 6. Calibrate the latency model

After flashing, the serial log prints `invoke N ms` per frame. Add one row per measured model to `latency_model.measurements` in `space.json`, fit, and paste the result back so `pred_ms` and the latency constraint become real.

In [ ]:
measurements = [
    # {"mmacs": 15.22, "ms": 0.0},   # e.g. the current deployed model
]
if measurements:
    print(td.models.fit_latency(measurements))